## Extracting the text and links from the PDF

In [2]:
import os

BASE_DIR = os.getcwd()
PARENT_DIR = os.path.dirname(BASE_DIR)

In [4]:
from langchain_community.document_loaders import PDFPlumberLoader, TextLoader

c:\Users\sayan\OneDrive\Desktop\Learning\innomatics-internship-projects\ai-resume-screening-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
def load_text(file_path: str) -> str: 
    if file_path.endswith(".pdf"):
        loader = PDFPlumberLoader(file_path)
    elif file_path.endswith(".txt"):
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        raise ValueError(f"Unsupported file type: {file_path}")

    documents = loader.load()
    if not documents:
        return ""

    text = "\n".join([doc.page_content.strip() for doc in documents if doc.page_content])

    return text

## Text Preprocessing 

Using Spacy to extract the keywords and phrases from the text.

In [6]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\.\:/\-]', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [7]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [8]:
def extract_data(text: str):
    doc = nlp(text)

    keywords = set()
    phrases = []

    for token in doc:
        if token.pos_ in ["NOUN", "PROPN", "ADJ"]:
            keywords.add(token.lemma_.lower())

    phrases = [
        chunk.text.lower() for chunk in doc.noun_chunks if len(chunk.text.split()) > 1
    ]

    phrases = list(filter(lambda phrase: phrase not in keywords, phrases))

    return {"phrases": phrases, "keywords": keywords}

## Embeddings

In [9]:
from sentence_transformers import SentenceTransformer

In [10]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 187.72it/s]


In [12]:
def batch_encode(text_list: list[str]):
    return model.encode(text_list)

## Scoring

In [27]:
import numpy as np
from numpy import ndarray, float32
from sklearn.metrics.pairwise import cosine_similarity

def get_score(resume_embeddings : list[ndarray], jd_embeddings: list[ndarray]) -> float:
    score_matrix = cosine_similarity(
        resume_embeddings, jd_embeddings
    )

    best_matches = np.max(score_matrix, axis=1)

    return float(best_matches.mean())


def get_keyword_score(resume_keywords: set[str], jd_keywords: set[str]) -> dict:
    matched_skills = resume_keywords & jd_keywords
    missing_skills = jd_keywords - resume_keywords

    keyword_score = len(matched_skills) / len(jd_keywords)

    return {
        "matched_skills" : matched_skills,
        "missing_skills" : missing_skills,
        "keyword_score" : keyword_score
    }

## Testing

In [ ]:
# Loading the File path

resume_path = os.path.join(PARENT_DIR, "docs", "RESUME.pdf")
jd_path = os.path.join(PARENT_DIR, "docs", "sample_jd.txt")

In [ ]:
# Extracting the text data from the files

resume_text = clean_text(load_text(resume_path))
jd_text = clean_text(load_text(jd_path))

In [ ]:
# Extracting the meaningful data from the text

resume_data = extract_data(resume_text)
jd_data = extract_data(jd_text)

resume_phrases = resume_data["phrases"]
resume_keywords = resume_data["keywords"]

jd_phrases = jd_data["phrases"]
jd_keywords = jd_data["keywords"]

In [24]:
# Getting the embeddings for Phrases

resume_embeddings = batch_encode(resume_phrases)
jd_embeddings = batch_encode(jd_phrases)

In [28]:
# Scoring the resume

phrase_score = get_score(resume_embeddings, jd_embeddings)

result = get_keyword_score(resume_keywords, jd_keywords)

keyword_score = result["keyword_score"]
matched_skills = result["matched_skills"]
missing_skills = result["missing_skills"]

In [29]:
final_score = (0.7 * phrase_score) + (0.3 * keyword_score)

In [33]:
print(f"""Final Score: {round(final_score * 100, 2)}
Phrase Score: {round(phrase_score * 100, 2)}
Keyword score: {round(keyword_score * 100, 2)}"""
)

Final Score: 44.58
Phrase Score: 45.83
Keyword score: 41.67


**Will get the AI Insights from the resume and job description and compare them to get a score.**

In [34]:
from langchain_core.prompts import PromptTemplate

In [35]:
template = """
You are an expert resume reviewer and career advisor.
Analyze the following resume vs job description data and give actionable improvement suggestions.


DATA:
- Final Match Score: {final_score}
- Phrase Score: {phrase_score}
- Keyword Score: {keyword_score}

- Matched Skills: {matched_skills}
- Missing Skills: {missing_skills}

- Resume Keywords: {resume_keywords}
- Job Description Keywords: {jd_keywords}


TASK:
1. Provide a detailed explanation of the final match score, explaining how well the resume aligns with the job description based on the final score.
2. Highlight the key weaknesses in the resume by analyzing the matched and missing skills.
3. Suggest specific, actionable improvements to enhance the resume, especially in terms of required skills, projects, or experience.
4. Recommend important skills that the candidate should focus on learning, based on the identified gaps.
5. Offer suggestions to improve the wording, clarity, and overall impact of the resume, including use of strong action verbs and measurable achievements.
6. Conclude with a clear and concise final verdict on the candidate’s suitability for the role.


OUTPUT FORMAT:
- Score Explanation:
- Weaknesses:
- Improvements:
- Recommended Skills:
- Resume Enhancement Tips:
- Final Verdict:


RESPONSE GUIDELINES:
- Be clear, concise, and professional  
- Avoid generic advice; make suggestions specific to the provided data  
- Focus on practical and actionable recommendations  
- Do not repeat the input data unnecessarily  
"""

In [36]:
prompt = PromptTemplate(
    input_variables=[
        "final_score",
        "phrase_score",
        "keyword_score",
        "matched_skills",
        "missing_skills",
        "resume_keywords",
        "jd_keywords"
    ],
    template=template
)

In [38]:
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [39]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0.3,
    model_name="llama-3.3-70b-versatile"
)

In [40]:
prompt = prompt.format(
    final_score=round(final_score * 100 , 2),
    phrase_score=round(phrase_score * 100 , 2),
    keyword_score=round(keyword_score * 100, 2),
    matched_skills=matched_skills,
    missing_skills=missing_skills,
    resume_keywords=resume_keywords,
    jd_keywords=jd_keywords
)

In [41]:
response = llm.invoke(prompt)
print(response.content)

**Score Explanation:**
The final match score of 44.58 indicates a moderate alignment between the resume and the job description. This score suggests that while the candidate has some relevant skills and experience, there are significant gaps that need to be addressed to make the resume more competitive. The phrase score of 45.83 and keyword score of 41.67 further emphasize the need for improvement in terms of using relevant phrases and keywords from the job description.

**Weaknesses:**
The matched skills show that the candidate has a good foundation in programming, machine learning, and data science. However, the missing skills highlight significant gaps in areas such as backend development, cloud computing, containerization, and software engineering. The candidate also lacks experience in deployment, testing, and debugging, which are crucial skills for the role.

**Improvements:**
To enhance the resume, the candidate should:
1. Emphasize backend development experience and skills, suc